# LLM miễn phí trên Colab T4 cho LegacyLens

Chạy Ollama với Qwen2.5 7B trên GPU T4 của Colab, mở một đường hầm Cloudflare để Codespaces gọi tới.

1. Runtime > Change runtime type > **T4 GPU**.
2. Runtime > Run all. Ô cuối in ra dòng `LLM_BASE_URL=...`.
3. Chép hai dòng đó vào `.env` trong Codespaces (hoặc Codespaces secrets), rồi chạy `make extract`.

Lưu ý: URL đường hầm không có mật khẩu, ai biết URL đều gọi được. Không chia sẻ URL và tắt runtime khi xong
(Runtime > Disconnect and delete runtime). Colab miễn phí tự ngắt khi để yên lâu, khi đó chạy lại từ đầu và cập nhật URL mới.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!sudo apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import os, subprocess, time

os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
os.environ["OLLAMA_CONTEXT_LENGTH"] = "32768"  # agent đọc nhiều đoạn code, cần cửa sổ dài
subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
!tail -n 3 ollama.log

In [ ]:
MODEL = "qwen2.5:7b-instruct"  # khoảng 4,7 GB, tải mất vài phút
!ollama pull {MODEL}

In [ ]:
# Thử một câu: phải in ra một câu trả lời ngắn
!curl -s http://localhost:11434/v1/chat/completions -H 'Content-Type: application/json' -d '{"model": "qwen2.5:7b-instruct", "messages": [{"role": "user", "content": "Say OK"}]}' | head -c 400

In [ ]:
import re

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:11434", "--http-host-header", "localhost:11434"],
    stdout=open("tunnel.log", "w"), stderr=subprocess.STDOUT,
)
url = None
for _ in range(30):
    time.sleep(2)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("tunnel.log").read())
    if m:
        url = m.group(0)
        break
print("Chép vào .env trong Codespaces:\n")
print(f"LLM_BASE_URL={url}/v1" if url else "Chưa có URL, xem tunnel.log")
print(f"LLM_MODEL={MODEL}")
print("LLM_API_KEY=")

Giữ tab này mở trong lúc chạy `make extract` và `make eval-qa`. Kiểm tra từ Codespaces:

```bash
curl -s $LLM_BASE_URL/models
```